In [ ]:
import pandas as pd
from pace_pic_validation.stats.regression import RobustRegression
import matplotlib.pyplot as plt
import glob
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
import re 
from shapely.geometry import Point
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
matchup_dir_lisstpic = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/matchups_lisstpic/allinone'
matchup_dir_discrete = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/matchups_discrete/240min_2km'
matchup_dir_discrete_modis = '/Users/timfugl/Desktop/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/matchups_discrete_modis'

In [ ]:


OBPG_L2_FLAGS = {
    0:  "ATMFAIL",
    1:  "LAND",
    2:  "PRODWARN",
    3:  "HIGLINT",
    4:  "HILT",
    5:  "HISATZEN",
    6:  "COASTZ",
    # 7: spare
    8:  "STRAYLIGHT",
    9:  "CLDICE",
    10: "COCCOLITH",
    11: "TURBIDW",
    12: "HISOLZEN",
    # 13: spare
    14: "LOWLW",
    15: "CHLFAIL",
    16: "NAVWARN",
    17: "ABSAER",
    # 18: spare
    19: "MAXAERITER",
    20: "MODGLINT",
    21: "CHLWARN",
    22: "ATMWARN",
    # 23: spare
    24: "SEAICE",
    25: "NAVFAIL",
    26: "FILTER",
    # 27: spare
    28: "BOWTIEDEL",
    29: "HIPOL",
    30: "PRODFAIL",
    # 31: spare
    }

def flag_mask(df, flag_column_name, drop, flag_list):
    """Mask the DataFrame based on a list of flags."""
    # Create a boolean mask for the specified flags
    mask = df[flag_column_name].isin(flag_list)
    
def CL2algorithm(rrs547,rrs667, old = True ):
    cl2 = rrs547 - rrs667

    if old:
        PIC = 0.4579 * cl2 -0.0006 #this is in mol/m3

    else: 
        PIC = 79.446*(cl2**2.160) # this is in ? 
        
    return PIC

COAST_PATH = (
    '/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/'
    'Obj1_PIC Product Validation for PACE/Data/Geo data/ne_10m_coastline.geojson'
)

def add_distance_to_coast(
    df,
    lon_col="lon_insitu",
    lat_col="lat_insitu",
    coast_path=COAST_PATH,
    target_crs="EPSG:4087",
    min_dist_km=None,
    distance_condition="greater",
    inplace=False,
):
    """
    Add a 'dist_to_coast_km' column and optionally filter by distance
    from the coastline.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame with longitude and latitude columns.

    lon_col : str
        Longitude column name.

    lat_col : str
        Latitude column name.

    coast_path : str
        Path to coastline GeoJSON or shapefile.

    target_crs : str
        Projected CRS with metre units for distance calculations.

    min_dist_km : float or None
        Distance threshold in kilometres.

    distance_condition : {"greater", "less"}
        Filtering condition:

        - "greater": keep points farther than min_dist_km
        - "less": keep points closer than min_dist_km

    inplace : bool
        If True, modify the input DataFrame. Otherwise, return a copy.

    Returns
    -------
    pd.DataFrame
        DataFrame with 'dist_to_coast_km' added and optionally filtered.
    """

    if not inplace:
        df = df.copy()

    if distance_condition not in {"greater", "less"}:
        raise ValueError(
            "distance_condition must be either 'greater' or 'less'"
        )

    if min_dist_km is not None and min_dist_km < 0:
        raise ValueError("min_dist_km must be non-negative")

    # Build point geometries
    geometry = [
        Point(lon, lat)
        for lon, lat in zip(df[lon_col], df[lat_col])
    ]

    points_gdf = gpd.GeoDataFrame(
        df,
        geometry=geometry,
        crs="EPSG:4326",
    )

    # Load coastline and assign geographic CRS if necessary
    coast = gpd.read_file(coast_path)

    if coast.crs is None:
        coast = coast.set_crs("EPSG:4326")
    else:
        coast = coast.to_crs("EPSG:4326")

    # Project both datasets to a metric CRS
    points_proj = points_gdf.to_crs(target_crs)
    coast_proj = coast.to_crs(target_crs)

    # Merge coastline geometries
    coast_union = coast_proj.union_all()

    # Calculate distance in metres, then convert to kilometres
    points_proj["dist_to_coast_km"] = (
        points_proj.geometry.distance(coast_union) / 1000.0
    )

    # Preserve the original DataFrame structure
    df["dist_to_coast_km"] = points_proj["dist_to_coast_km"].to_numpy()

    # Optional filtering
    if min_dist_km is not None:
        if distance_condition == "greater":
            mask = df["dist_to_coast_km"] > min_dist_km
        else:
            mask = df["dist_to_coast_km"] < min_dist_km

        df = df.loc[mask].copy()

    return df

def plot_sat_insitu_panel(
    ax,
    x,
    y,
    y_pred,
    r2,
    rmse,
    mae,
    model_str,
    title,
    xlabel="PIC discrete [mmol/m³]",
    ylabel="CI2 PIC (PACE) [mmol/m³]",
    scatter_color="red",
    line_color="blue",
    s=15,
    alpha=0.8,
    edgecolor="k",
    show_grid=True,
):
    """
    Plot a single satellite vs in situ panel on a given Axes object.
    """

    # Convert to arrays and identify the largest plotted x/y value
    x = np.asarray(x)
    y = np.asarray(y)
    y_pred = np.asarray(y_pred)

    finite_values = np.concatenate([
        x[np.isfinite(x)],
        y[np.isfinite(y)],
    ])

    if finite_values.size == 0:
        raise ValueError("x and y contain no finite values.")

    axis_max = np.nanmax(finite_values) + 0.5

    # Use zero as the lower limit.
    axis_min = 0

    # Basic axes setup
    if show_grid:
        ax.grid(True, which="both", ls=":", lw=0.5)

    ax.set_xscale("linear")
    ax.set_yscale("linear")
    ax.minorticks_off()

    # Scatter
    ax.scatter(
        x,
        y,
        color=scatter_color,
        s=s,
        alpha=alpha,
        edgecolors=edgecolor,
        lw=0.5,
    )

    # Use the full axis range for the regression line
    slope, intercept = np.polyfit(x, y_pred, deg=1)

    x_line = np.linspace(axis_min, axis_max, 200)
    y_line = slope * x_line + intercept

    ax.plot(
        x_line,
        y_line,
        label=(
            f"R² = {r2:.4f}\n"
            f"{model_str}\n"
            f"RMSE = {rmse:.3f}\n"
            f"MAE = {mae:.3f}"
        ),
        color=line_color,
        linestyle="-",
        lw=1,
)

    # 1:1 reference line
    ax.plot(
        [axis_min, axis_max],
        [axis_min, axis_max],
        color="black",
        linestyle="--",
        linewidth=1,
        label="1:1 line",
    )

    # Identical x and y limits
    ax.set_xlim(axis_min, axis_max)
    ax.set_ylim(axis_min, axis_max)

    # One unit on x has the same visual length as one unit on y
    ax.set_aspect("equal", adjustable="box")

    ax.legend(framealpha=0.3)

    # Labels and title
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

def plot_pic_map_and_histogram(
    df,
    lat_col="lat",
    lon_col="lon",
    datetime_col="datetime",
    pic_col="PIC_mmolC_m3",
    title=None,
    cmin=0.0,
    cmax=4.0,
    nbins=50,
    height=600,
    map_style="carto-positron",
    map_zoom=1,
    colorscale="Viridis",
):
    """
    Create a two-panel Plotly figure:
      - Left: map of PIC observations (Scattermap)
      - Right: histogram of PIC values

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with lat, lon, datetime, and PIC columns.
    lat_col : str
        Column name for latitude.
    lon_col : str
        Column name for longitude.
    datetime_col : str
        Column name for datetime (used in hover).
    pic_col : str
        Column name for PIC concentration.
    title : str or None
        Optional figure title.
    cmin, cmax : float
        Color scale limits for PIC on the map.
    nbins : int
        Number of bins for the histogram.
    height : int
        Figure height in pixels.
    map_style : str
        Map style for Scattermap (e.g. "carto-positron").
    map_zoom : int or float
        Initial map zoom.
    colorscale : str or list
        Colorscale for PIC markers.

    Returns
    -------
    fig : plotly.graph_objects.Figure
        The generated figure.
    """

    fig = make_subplots(
        rows=1,
        cols=2,
        column_widths=[0.65, 0.35],
        specs=[[{"type": "scattermap"}, {"type": "xy"}]],
        subplot_titles=("PIC locations", "PIC distribution"),
    )

    # Map trace
    fig.add_trace(
        go.Scattermap(
            lat=df[lat_col],
            lon=df[lon_col],
            mode="markers",
            marker=dict(
                size=7,
                color=df[pic_col],
                colorscale=colorscale,
                showscale=True,
                colorbar=dict(title="PIC"),
                cmin=cmin,
                cmax=cmax,
            ),
            customdata=df[[datetime_col, pic_col, lat_col, lon_col]],
            hovertemplate=(
                "Datetime: %{customdata[0]|%Y-%m-%d %H:%M:%S}<br>"
                "PIC: %{customdata[1]:.3g}<br>"
                "Lat: %{customdata[2]:.3f}<br>"
                "Lon: %{customdata[3]:.3f}<br>"
                "<extra></extra>"
            ),
            name="Observations",
        ),
        row=1,
        col=1,
    )

    # Histogram trace
    fig.add_trace(
        go.Histogram(
            x=df[pic_col],
            nbinsx=nbins,
            marker=dict(color="steelblue"),
            name="PIC histogram",
        ),
        row=1,
        col=2,
    )

    fig.update_layout(
        height=height,
        map=dict(style=map_style, zoom=map_zoom),
        margin=dict(l=20, r=20, t=50, b=20),
        hovermode="closest",
    )
    if title is not None:
        fig.update_layout(title=dict(text=title, x=0.5))

    fig.update_xaxes(title_text="PIC", row=1, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=2)

    return fig

import re
import pandas as pd

def filter_flag_column(
    df,
    flag_col,
    exclude_flags=None,
    allow_only_flags=None,
    inplace=False,
    return_mask=False,
    rrs_cols=None,
    mode="drop_row",
):
    """
    Filter rows or set NaN in specific columns based on decoding flags.

    Supports three modes:
      - "drop_row": drop rows where any exclude_flags are present.
      - "set_nan": keep all rows, but set specified rrs_cols to NaN where exclude flags are triggered.
      - "allow_only": keep only rows where flags are either empty or all in allow_only_flags.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    flag_col : str
        Column name containing flag strings like "['A', 'B']".
    exclude_flags : list[str], optional
        List of flag strings to exclude. Required for mode in {"drop_row", "set_nan"}.
    allow_only_flags : list[str], optional
        List of flag strings that are allowed. Used only for mode="allow_only".
        Rows are kept if:
          - flags are empty/NaN, OR
          - all non-empty flags are in allow_only_flags.
    inplace : bool
        If True, modify df in place; otherwise return a new DataFrame.
    return_mask : bool
        If True, also return the boolean mask used for filtering.
    rrs_cols : list[str], optional
        Column names to set to NaN where flags are triggered.
        Required if mode="set_nan".
    mode : {"drop_row", "set_nan", "allow_only"}
        - "drop_row": drop rows where any exclude_flags are present (original behavior).
        - "set_nan": keep all rows, but set specified rrs_cols to NaN where exclude flags are triggered.
        - "allow_only": keep only rows with no flags or only flags in allow_only_flags.

    Returns
    -------
    df_out : pd.DataFrame
        Filtered or modified DataFrame.
    mask : pd.Series (optional)
        Boolean mask:
          - mode="drop_row": True = keep row.
          - mode="set_nan": True = row had exclude flags (NaN was applied).
          - mode="allow_only": True = row is kept.
    """
    if mode not in ("drop_row", "set_nan", "allow_only"):
        raise ValueError("mode must be 'drop_row', 'set_nan', or 'allow_only'")

    if mode in ("drop_row", "set_nan"):
        if exclude_flags is None:
            raise ValueError("exclude_flags must be provided for mode='drop_row' or 'set_nan'")
    if mode == "set_nan" and rrs_cols is None:
        raise ValueError("rrs_cols must be provided when mode='set_nan'")
    if mode == "allow_only" and allow_only_flags is None:
        raise ValueError("allow_only_flags must be provided when mode='allow_only'")

    if not inplace:
        df = df.copy()

    if exclude_flags is not None:
        exclude_flags = set(exclude_flags)
    if allow_only_flags is not None:
        allow_only_flags = set(allow_only_flags)

    pattern = re.compile(r"['\"]([^'\"]+)['\"]")

    def parse_flags(val):
        if val is None or (isinstance(val, float) and val != val):  # NaN
            return set()
        if isinstance(val, (list, tuple, set)):
            return set(val)
        s = str(val)
        tokens = pattern.findall(s)
        return set(tokens)

    def has_exclude_flags(flags_raw):
        flags = parse_flags(flags_raw)
        return len(flags & exclude_flags) > 0

    def is_allowed_row(flags_raw):
        flags = parse_flags(flags_raw)
        if len(flags) == 0:
            return True
        return flags <= allow_only_flags

    if mode == "drop_row":
        mask_exclude = df[flag_col].apply(has_exclude_flags)
        mask_keep = ~mask_exclude
        df = df[mask_keep].reset_index(drop=True)
        if return_mask:
            return df, mask_keep
        return df

    elif mode == "set_nan":
        mask_exclude = df[flag_col].apply(has_exclude_flags)
        df.loc[mask_exclude, rrs_cols] = pd.NA
        if return_mask:
            return df, mask_exclude
        return df

    elif mode == "allow_only":
        mask_keep = df[flag_col].apply(is_allowed_row)
        df = df[mask_keep].reset_index(drop=True)
        if return_mask:
            return df, mask_keep
        return df

In [ ]:
# creating matchup dfs

insitu_all = '/Volumes/Extreme SSD/PhD backup/PhD_Ocean_Color_Chemistry/Obj1_PIC Product Validation for PACE/Data/PIC_discrete/all_PIC.csv'




discrete_list = glob.glob(matchup_dir_discrete + '/*.csv')
lisstpic_list = glob.glob(matchup_dir_lisstpic + '/*.csv')
discrete_list_modis = glob.glob(matchup_dir_discrete_modis + '/*.csv')

discrete = pd.concat((pd.read_csv(f) for f in discrete_list))

discrete.reset_index()

discrete.to_csv('/Users/timfugl/Desktop/untitled folder 2/discrete_pace_matchups.csv')
discrete_modis = pd.concat((pd.read_csv(f) for f in discrete_list_modis))

lisstpic = pd.concat((pd.read_csv(f) for f in lisstpic_list))


insitu = pd.read_csv(insitu_all)

insitu = insitu[~insitu['Station'].str.startswith('CTD')]

insitu = insitu.reset_index()

remove_G = False 

if remove_G:

    print(len(discrete))
    to_drop = []
    for label, idx in discrete['insitu_idx'].items():  # label = discrete.index at that row
        idx = int(idx)
        station_value = insitu['Station'].iloc[idx]
        if isinstance(station_value, str) and station_value.startswith('G'):
            to_drop.append(label)  # use label, not i

    discrete = discrete.drop(to_drop, errors='ignore').reset_index(drop=True)
    print(len(discrete))

print(discrete.columns)



In [ ]:
dfs = [discrete, lisstpic, discrete_modis]
names = ['discrete', 'lisstpic', 'discrete']
satellites = ['PACE', 'PACE', 'MODIS']

print(discrete_modis)

'''
- land

- cloud or ice

- stray light

- sun glint

- high top-of-atmosphere (TOA) radiance, i.e. at or near saturation

- low Lwn(555)L_{wn}(555)Lwn​(555) (used to identify cloud-shadowed pixels)

- atmospheric correction failure

    0:  "ATMFAIL",
    1:  "LAND",
    2:  "PRODWARN",
    3:  "HIGLINT",
    4:  "HILT",
    5:  "HISATZEN",
    6:  "COASTZ",
    # 7: spare
    8:  "STRAYLIGHT",
    9:  "CLDICE",
    10: "COCCOLITH",
    11: "TURBIDW",
    12: "HISOLZEN",
    # 13: spare
    14: "LOWLW",
    15: "CHLFAIL",
    16: "NAVWARN",
    17: "ABSAER",
    # 18: spare
    19: "MAXAERITER",
    20: "MODGLINT",
    21: "CHLWARN",
    22: "ATMWARN",
    # 23: spare
    24: "SEAICE",
    25: "NAVFAIL",
    26: "FILTER",
    # 27: spare
    28: "BOWTIEDEL",
    29: "HIPOL",
    30: "PRODFAIL",

'''

# filter parameters
min_dist_km = 75
distance_condition = 'less' # or greater
exclude = ['PRODFAIL',"ATMFAIL",  "LAND", "HIGLINT", "STRAYLIGHT"]
include = ["MODGLINT","COCCOLITH"] #center pixel only 
use_flag = False
cv_max = 0.15 # cv of 3x3 pixel box 
valid_pixels_min = 5
max_pic_value = 1000



for df, name, satellite in zip(dfs, names, satellites):

    print('no filter')
    print(len(df['granule'].unique()))


   
    if name == 'penis': 

        print(f'length {len(df)} before grouping')
        
        idx = df.groupby("granule")["spatial_dist_km"].idxmin()
        df = df.loc[idx].reset_index(drop=True)
     
        print(f'length {len(df)} after grouping')
        





    df = df[~df['Rrs_547_mean'].isna()]

    print('after PRODFAIL')

    print(len(df['granule'].unique()))
    
    df = df[df['insitu_value']< max_pic_value]

    print('after distance filter')
    print(len(df['granule'].unique()))

    
    df =  add_distance_to_coast(
        df,
        lon_col='lon_insitu',
        lat_col='lat_insitu',
        coast_path=COAST_PATH,
        target_crs="EPSG:4087",
        min_dist_km=min_dist_km,
        inplace=True,
        distance_condition = distance_condition
    )


    if use_flag:

        # filtering flags - direct matchup pixel
        print(f'before px4 (center) flag filtering: {len(df['granule'].unique())} match-ups.')

        df = filter_flag_column(
            df,
            flag_col="flag_px_decode4",
            exclude_flags = exclude,  # whatever you need
            allow_only_flags = None,
            mode= 'drop_row',  #'drop_row', 'set_nan', 'allow_only'
            inplace = True,
        )

        print(f'after flag px4 (center) filtering: {len(df['granule'].unique())} match-ups.')
        # filtering flag - setting nan for invalid pixels in pixel box 

        for i in range(0,9):
            df = filter_flag_column(
                df,
                flag_col=f"flag_px_decode{i}",
                exclude_flags=exclude,
                rrs_cols=[f"Rrs_547_px{i}_raw", f"Rrs_667_px{i}_raw"],
                mode="set_nan",
            )


            

        # counting nan rrs values 

        # pixels = list(range(0, 9))  # 0–8, once you have all columns
        pixels = list(range(0, 9))  # 0–8
        bands = ["Rrs_547", "Rrs_667"]

        rrs_cols = [f"{band}_px{p}_raw" for p in pixels for band in bands]
        rrs_cols = [c for c in rrs_cols if c in df.columns]

        print(f'before valid px filtering: {len(df)} match-ups.')
            # Count non-NA values per row across those columns
        df["n_valid_pixels"] = df[rrs_cols].notna().sum(axis=1)
            # masking all pixels with flags in the 3x3 pixel box

        df = df[df['n_valid_pixels']>= valid_pixels_min]

        print(f'after valid px filtering: {len(df)} match-ups.')

    # filtering CV of 3x3 pixel box
    print(f'before CV {cv_max} filtering: {len(df)} match-ups.')
    df = df[df['Rrs_667_cv'] < cv_max]
    print(f'after CV {cv_max} filtering: {len(df)} match-ups.')


    # calculating Sat PIC
    # old algorithm

    print(f'calculating ci2 for {len(df['Rrs_547_px4_raw'].unique())} values')
    df['CI2_PIC_mmol_m3_old'] = CL2algorithm(df['Rrs_547_px4_raw'], df['Rrs_667_px4_raw'], old = True) * 1000



    # new algorithm
    df['CI2_PIC_mmol_m3_new'] = CL2algorithm(df['Rrs_547_px4_raw'], df['Rrs_667_px4_raw'], old = False) * 1000

    # second na (necessary bc 3 random nans in rrs 547. idk why they are not dropped before)
    df = df.dropna(subset = ['insitu_value', 'Rrs_547_px4_raw'])
    # robust regression for old and new models
    y_pred_old, model_old, intercept_old, coefficient_old, r2_old, rmse_old, mae_old = RobustRegression(df['insitu_value'], df['CI2_PIC_mmol_m3_old'])
    y_pred_new, model_new, intercept_new, coefficient_new, r2_new, rmse_new, mae_new = RobustRegression(df['insitu_value'], df['CI2_PIC_mmol_m3_new'])

    # show in-situ on map and ranges 

    fig = plot_pic_map_and_histogram(
        df,
        lat_col="lat_insitu",
        lon_col="lon_insitu",
        datetime_col="insitu_timestamp",
        pic_col="insitu_value",
        title="Match-ups – locations and distribution",
        cmin=0.0,
        cmax=4.0,
        nbins=50,
    )

    fig.show()

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))

    # Panel A – old algorithm
    plot_sat_insitu_panel(
        ax=axs[0],
        x=df['insitu_value'],
        y=df["CI2_PIC_mmol_m3_old"],
        y_pred=y_pred_old,
        r2=r2_old,
        rmse=rmse_old,
        mae=mae_old,
        model_str=model_old,
        title=f"A: {satellite} vs {name} samples, linear CI2",
        xlabel="PIC discrete [mmol/m³]",
        ylabel="CI2 PIC (SAT) [mmol/m³]",
        scatter_color="red",
    )

    # Panel B – new algorithm
    plot_sat_insitu_panel(
        ax=axs[1],
        x=df['insitu_value'],
        y=df["CI2_PIC_mmol_m3_new"],  # or whatever your new y is
        y_pred=y_pred_new,
        r2=r2_new,
        rmse=rmse_new,
        mae=mae_new,
        model_str=model_new,
        title=f"B: {satellite} vs {name} samples, power law CI2",
        xlabel="PIC discrete [mmol/m³]",
        ylabel="CI2 PIC (SAT) [mmol/m³]",
        scatter_color="red",
    )

    plt.tight_layout()
    plt.show()